# Розвідувальний Аналіз Даних (EDA)
## Датасет: База даних американських гірок (Roller Coaster Database)

---

> **Що таке EDA?**  
> Розвідувальний аналіз даних (Exploratory Data Analysis) — це процес дослідження набору даних з метою виявлення основних характеристик, закономірностей, аномалій та зв'язків між змінними. EDA передує будь-якому формальному моделюванню і є фундаментом якісного аналізу.

### Кроки EDA, які ми пройдемо:
1. Завантаження та початковий огляд даних
2. Дослідження структури та типів даних
3. Обробка пропущених значень
4. Аналіз характеристик даних
5. Трансформація даних
6. Візуалізація залежностей
7. Виявлення та обробка викидів
8. Висновки та інсайти
9. Корисні однорядкові функції для студентів

---
## Крок 0: Встановлення та імпорт бібліотек

Перед початком роботи нам потрібно встановити `kagglehub` для завантаження датасету та імпортувати всі необхідні бібліотеки.

- **pandas** — для роботи з табличними даними
- **numpy** — для математичних операцій
- **matplotlib / seaborn** — для візуалізації
- **kagglehub** — для завантаження датасетів з Kaggle

In [ ]:
# Встановлення kagglehub (якщо ще не встановлено)
!pip install kagglehub -q

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Глобальні налаштування візуалізації
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid', palette='muted')

print('Всі бібліотеки успішно імпортовані!')

---
## Крок 1: Завантаження та початковий огляд даних

**Навіщо?** Перш ніж аналізувати — потрібно завантажити дані і зрозуміти їх загальну структуру. На цьому кроці ми дізнаємося:
- скільки рядків і стовпців у датасеті
- як виглядають перші/останні записи
- які назви колонок

Датасет містить інформацію про **1087 американських гірок** з усього світу: назву, розташування, швидкість, висоту, статус, виробника тощо.

In [ ]:
# Завантаження датасету з Kaggle
path = kagglehub.dataset_download('robikscube/rollercoaster-database')
print(f'Шлях до файлів датасету: {path}')

In [ ]:
import os

# Перегляд файлів у директорії датасету
files = os.listdir(path)
print('Файли в директорії датасету:')
for f in files:
    print(f'  - {f}')

In [ ]:
# Завантажуємо основний CSV файл
csv_file = [f for f in files if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

print(f'Датасет завантажено: "{csv_file}"')
print(f'Розмір: {df.shape[0]} рядків × {df.shape[1]} стовпців')

In [ ]:
# Перші 5 рядків датасету
print('Перші 5 рядків датасету:')
df.head()

In [ ]:
# Останні 5 рядків датасету
print('Останні 5 рядків датасету:')
df.tail()

In [ ]:
# Список усіх колонок
print('Список усіх колонок:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:>2}. {col}')

---
## Крок 2: Дослідження структури та типів даних

**Призначення кроку** Розуміння типів даних критично важливе:
- числові дані (`int64`, `float64`) — можна рахувати середнє, медіану тощо
- текстові дані (`object`) — потребують кодування або окремої обробки
- Неправильні типи (наприклад, дата у форматі рядка) — потрібно конвертувати

На додачу буде досліджено ункікальні значення та переглянуто базову статистику

In [ ]:
# Детальна інформація про датасет: типи, кількість не-null значень
print('Загальна інформація про датасет:')
df.info()

In [ ]:
# Кількість унікальних значень у кожній колонці
print('Кількість унікальних значень у кожній колонці:')
df.nunique().sort_values(ascending=False)

In [ ]:
# Базова статистика для числових колонок
print('Базова описова статистика (числові колонки):')
df.describe().round(2)

In [ ]:
# Базова статистика для категоріальних колонок
print('Базова описова статистика (категоріальні колонки):')
df.describe(include='object')

In [ ]:
# Розподіл типів даних
dtype_counts = df.dtypes.value_counts()
print('Розподіл типів даних:')
for dtype, count in dtype_counts.items():
    print(f'  {str(dtype):<12}: {count} колонок')

---
## Крок 3: Аналіз та обробка пропущених значень

**Навіщо?** Реальні дані майже завжди містять пропуски (NaN, None). Пропущені значення можуть:
- спотворювати результати аналізу
- викликати помилки в алгоритмах машинного навчання
- вказувати на проблеми зі збором даних

**Стратегії обробки:**
- `fillna(значення)` — заповнити середнім/медіаною/модою або константою
- `dropna()` — видалити рядки/стовпці з пропусками
- Залишити (якщо пропусків мало і вони не критичні)

In [ ]:
# Підрахунок та відсоток пропущених значень
missing = pd.DataFrame({
    'Пропущено (к-сть)': df.isnull().sum(),
    'Пропущено (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['Пропущено (к-сть)'] > 0].sort_values('Пропущено (%)', ascending=False)

print(f'Колонок з пропусками: {len(missing)} з {len(df.columns)}')
print(f'Загальна кількість пропусків: {df.isnull().sum().sum()}')
missing

In [ ]:
# Теплова карта пропущених значень
fig, ax = plt.subplots(figsize=(16, 8))

# Вибираємо лише колонки з пропусками
cols_with_missing = missing.index.tolist()
missing_matrix = df[cols_with_missing].isnull()

sns.heatmap(
    missing_matrix.T,
    cmap='RdYlGn_r',
    cbar_kws={'label': '1 = Пропуск'},
    ax=ax,
    yticklabels=True
)
ax.set_title('Теплова карта пропущених значень', 
             fontsize=14, pad=15)
ax.set_xlabel('Індекс рядка')
ax.set_ylabel('Колонки')
plt.tight_layout()
plt.show()

In [ ]:
# Візуалізація відсотку пропусків по колонках
fig, ax = plt.subplots(figsize=(12, 8))

colors = ['#e74c3c' if x > 50 else '#f39c12' if x > 20 else '#3498db' 
          for x in missing['Пропущено (%)']]

bars = ax.barh(missing.index, missing['Пропущено (%)'], color=colors, edgecolor='white')

# Додаємо підписи на бари
for bar, val in zip(bars, missing['Пропущено (%)']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)

ax.axvline(x=50, color='red', linestyle='--', alpha=0.7, label='50% поріг')
ax.axvline(x=20, color='orange', linestyle='--', alpha=0.7, label='20% поріг')
ax.set_title('Відсоток пропущених значень по колонках', fontsize=14, pad=15)
ax.set_xlabel('Відсоток пропущених значень (%)')
ax.legend()
plt.tight_layout()
plt.show()

print()
print('Червоний (>50%): критично багато пропусків')
print('Жовтий (20-50%): суттєво — потребує уважної обробки')
print('Синій (<20%): прийнятно — можна заповнити')

In [ ]:
# Стратегія обробки пропусків для ключових числових колонок
# Для аналізу зосередимось на очищених числових колонках

key_numeric = ['speed_mph', 'height_ft', 'Inversions_clean', 'Gforce_clean', 'year_introduced']

print('Пропуски в ключових числових колонках:')
for col in key_numeric:
    if col in df.columns:
        n_missing = df[col].isnull().sum()
        pct = n_missing / len(df) * 100
        print(f'  {col:<20}: {n_missing:>4} ({pct:.1f}%)')

In [ ]:
# Створюємо робочу копію датасету
df_clean = df.copy()

# Заповнюємо пропуски медіаною для числових колонок
for col in ['speed_mph', 'height_ft', 'Gforce_clean']:
    if col in df_clean.columns:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f'{col}: пропуски заповнені медіаною = {median_val:.2f}')

# Видаляємо рядки, де відсутній рік введення в експлуатацію
before = len(df_clean)
df_clean = df_clean.dropna(subset=['year_introduced'])
after = len(df_clean)
print(f'Видалено {before - after} рядків без року введення')

print(f'\nРозмір після обробки: {df_clean.shape[0]} рядків × {df_clean.shape[1]} стовпців')

---
## Крок 4: Аналіз характеристик даних (Univariate Analysis)

**Навіщо?** Одновимірний аналіз (univariate) досліджує **кожну колонку окремо**. Це допомагає зрозуміти:
- розподіл числових даних (нормальний, скошений, бімодальний?)
- найпоширеніші категорії в категоріальних колонках
- наявність екстремальних значень

Основні інструменти: гістограми, boxplot, countplot, describe().

In [ ]:
# Розподіл числових змінних — гістограми з KDE
numeric_cols = ['speed_mph', 'height_ft', 'Inversions_clean', 'Gforce_clean', 'year_introduced']
numeric_cols = [c for c in numeric_cols if c in df_clean.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    data = df_clean[col].dropna()
    sns.histplot(data, kde=True, ax=axes[i], color='steelblue', edgecolor='white')
    axes[i].set_title(f'Розподіл: {col}', fontsize=13)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Кількість')

    # Додаємо лінії середнього і медіани
    axes[i].axvline(data.mean(), color='red', linestyle='--', label=f'Середнє: {data.mean():.1f}')
    axes[i].axvline(data.median(), color='green', linestyle='--', label=f'Медіана: {data.median():.1f}')
    axes[i].legend(fontsize=9)

# Вимикаємо зайву комірку
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Розподіл числових змінних', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Аналіз категоріальних змінних — топ-10 значень
cat_cols = ['Type_Main', 'Status', 'Manufacturer']
cat_cols = [c for c in cat_cols if c in df_clean.columns]

fig, axes = plt.subplots(1, len(cat_cols), figsize=(18, 6))
if len(cat_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, cat_cols):
    top10 = df_clean[col].value_counts().head(10)
    colors = sns.color_palette('Set2', len(top10))
    top10.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
    ax.set_title(f'Топ-10: {col}', fontsize=13)
    ax.set_xlabel('Кількість')
    ax.invert_yaxis()
    
    # Підписи на барах
    for i, (val, cnt) in enumerate(top10.items()):
        ax.text(cnt + 0.5, i, str(cnt), va='center', fontsize=10)

plt.suptitle('Найпоширеніші категорії', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Динаміка відкриття гірок по роках
fig, ax = plt.subplots(figsize=(16, 5))

yearly = df_clean['year_introduced'].value_counts().sort_index()
ax.bar(yearly.index, yearly.values, color='steelblue', edgecolor='white', alpha=0.85)
ax.plot(yearly.index, yearly.rolling(5).mean(), color='red', linewidth=2.5, label='Ковзне середнє (5 років)')

ax.set_title('Кількість нових гірок по роках', fontsize=14, pad=12)
ax.set_xlabel('Рік')
ax.set_ylabel('Кількість')
ax.legend()

# Підсвічуємо пік
peak_year = yearly.idxmax()
peak_val = yearly.max()
ax.annotate(f'Пік: {peak_year}\n({peak_val} гірок)',
            xy=(peak_year, peak_val), xytext=(peak_year - 15, peak_val - 5),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=11, color='red')

plt.tight_layout()
plt.show()

---
## Крок 5: Трансформація даних

**Навіщо?** Часто дані потребують перетворення перед аналізом:
- **Конвертація дат** з рядка у формат datetime
- **Нормалізація / стандартизація** числових змінних
- **Бінаризація** — перетворення числових значень у категорії
- **Логарифмічне перетворення** — для скошених розподілів

Ці перетворення покращують якість аналізу і підготовлюють дані до ML-моделей.

In [ ]:
# 1. Конвертація дати відкриття
if 'opening_date_clean' in df_clean.columns:
    df_clean['opening_date_clean'] = pd.to_datetime(df_clean['opening_date_clean'], errors='coerce')
    df_clean['opening_month'] = df_clean['opening_date_clean'].dt.month
    df_clean['opening_season'] = df_clean['opening_month'].map({
        12: 'Зима', 1: 'Зима', 2: 'Зима',
        3: 'Весна', 4: 'Весна', 5: 'Весна',
        6: 'Літо', 7: 'Літо', 8: 'Літо',
        9: 'Осінь', 10: 'Осінь', 11: 'Осінь'
    })
    print('Дата відкриття конвертована в datetime')
    print('Додано колонки: opening_month, opening_season')

# 2. Категоризація швидкості
if 'speed_mph' in df_clean.columns:
    df_clean['speed_category'] = pd.cut(
        df_clean['speed_mph'],
        bins=[0, 30, 50, 70, 100, 200],
        labels=['Дуже повільна', 'Повільна', 'Середня', 'Швидка', 'Екстремальна'],
        right=True
    )
    print('\nКатегоризація швидкості:')
    print(df_clean['speed_category'].value_counts())

# 3. Логарифмічне перетворення для скошених змінних
if 'speed_mph' in df_clean.columns:
    df_clean['log_speed'] = np.log1p(df_clean['speed_mph'])
    df_clean['log_height'] = np.log1p(df_clean['height_ft'])
    print('\nЛогарифмічні версії speed_mph та height_ft створено')

In [ ]:
# Порівняння оригінального та логарифмічного розподілу швидкості
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_clean['speed_mph'].dropna(), kde=True, ax=axes[0], color='coral')
axes[0].set_title('Оригінальний розподіл speed_mph', fontsize=13)
axes[0].set_xlabel('Швидкість (mph)')

sns.histplot(df_clean['log_speed'].dropna(), kde=True, ax=axes[1], color='mediumseagreen')
axes[1].set_title('Логарифмічний розподіл log(speed_mph)', fontsize=13)
axes[1].set_xlabel('log(Швидкість + 1)')

plt.suptitle('Ефект логарифмічного перетворення', fontsize=14)
plt.tight_layout()
plt.show()
print('Логарифмічне перетворення робить скошений розподіл більш симетричним')

In [ ]:
# Кількість гірок за сезоном відкриття
if 'opening_season' in df_clean.columns:
    season_order = ['Весна', 'Літо', 'Осінь', 'Зима']
    season_colors = {'Весна': '#2ecc71', 'Літо': '#f39c12', 'Осінь': '#e67e22', 'Зима': '#3498db'}
    
    season_counts = df_clean['opening_season'].value_counts().reindex(season_order).dropna()
    
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(season_counts.index, season_counts.values, 
                  color=[season_colors[s] for s in season_counts.index], edgecolor='white')
    
    for bar, val in zip(bars, season_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(val), ha='center', fontsize=12, fontweight='bold')
    
    ax.set_title('Відкриття гірок за сезоном', fontsize=14)
    ax.set_ylabel('Кількість')
    plt.tight_layout()
    plt.show()

---
## Крок 6: Візуалізація залежностей (Bivariate & Multivariate Analysis)

**Навіщо?** Двовимірний і багатовимірний аналіз дозволяє виявити:
- **кореляції** між числовими змінними (чи пов'язані вони?)
- **відмінності** між категоріями (чи відрізняється швидкість для різних типів гірок?)
- **комплексні закономірності**, що не видно при аналізі кожної колонки окремо

Інструменти: scatterplot, boxplot, heatmap кореляцій, pairplot.

In [ ]:
# Кореляційна матриця числових змінних
corr_cols = ['speed_mph', 'height_ft', 'Inversions_clean', 'Gforce_clean', 'year_introduced']
corr_cols = [c for c in corr_cols if c in df_clean.columns]

corr_matrix = df_clean[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Маскуємо верхній трикутник

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    mask=mask,
    ax=ax,
    square=True,
    linewidths=1
)
ax.set_title('Кореляційна матриця числових змінних\n(від -1 до +1, де 1 = повна пряма залежність)', 
             fontsize=13, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: Висота vs Швидкість (з розмаїттям за типом)
fig, ax = plt.subplots(figsize=(12, 7))

if 'Type_Main' in df_clean.columns:
    top_types = df_clean['Type_Main'].value_counts().head(4).index
    plot_data = df_clean[df_clean['Type_Main'].isin(top_types)]
    
    palette = {'Steel': '#3498db', 'Wood': '#e67e22', 'Other': '#95a5a6', 'Hybrid': '#9b59b6'}
    
    for type_name, group in plot_data.groupby('Type_Main'):
        color = palette.get(type_name, '#cccccc')
        ax.scatter(group['height_ft'], group['speed_mph'],
                   label=type_name, alpha=0.6, s=60, color=color, edgecolors='white', linewidth=0.5)
else:
    ax.scatter(df_clean['height_ft'], df_clean['speed_mph'], alpha=0.5, s=50)

ax.set_xlabel('Висота (футів)', fontsize=12)
ax.set_ylabel('Швидкість (mph)', fontsize=12)
ax.set_title('Залежність швидкості від висоти гірки (за типом)', fontsize=14, pad=12)
ax.legend(title='Тип', fontsize=11)
plt.tight_layout()
plt.show()
print('Чим вища гірка — тим, як правило, швидша. Ця залежність є фізичною закономірністю.')

In [ ]:
# Boxplot: Розподіл швидкості за типом гірки
if 'Type_Main' in df_clean.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    type_order = df_clean.groupby('Type_Main')['speed_mph'].median().sort_values(ascending=False).index
    
    sns.boxplot(
        data=df_clean,
        x='Type_Main',
        y='speed_mph',
        order=type_order,
        palette='Set2',
        width=0.6,
        ax=ax
    )
    ax.set_title('Розподіл швидкості за типом гірки (відсортовано за медіаною)', fontsize=13, pad=12)
    ax.set_xlabel('Тип гірки')
    ax.set_ylabel('Швидкість (mph)')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()

In [ ]:
# Pairplot для числових змінних
pairplot_cols = ['speed_mph', 'height_ft', 'Inversions_clean', 'Gforce_clean']
pairplot_cols = [c for c in pairplot_cols if c in df_clean.columns]

if 'Type_Main' in df_clean.columns:
    top2 = df_clean['Type_Main'].value_counts().head(2).index
    pair_data = df_clean[df_clean['Type_Main'].isin(top2)][pairplot_cols + ['Type_Main']].dropna()
    
    pair_data_sample = pair_data.sample(min(300, len(pair_data)), random_state=42)
    
    g = sns.pairplot(pair_data_sample, hue='Type_Main', diag_kind='kde', 
                     plot_kws={'alpha': 0.5, 's': 30}, palette='husl')
    g.figure.suptitle('Pairplot: взаємозалежності між числовими змінними', 
                      y=1.02, fontsize=14)
    plt.show()

In [ ]:
# Топ-15 виробників за середньою швидкістю гірок
if 'Manufacturer' in df_clean.columns:
    top_manufacturers = df_clean.groupby('Manufacturer').agg(
        avg_speed=('speed_mph', 'mean'),
        count=('speed_mph', 'count')
    ).query('count >= 5').sort_values('avg_speed', ascending=False).head(15)
    
    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(top_manufacturers.index, top_manufacturers['avg_speed'],
                   color=sns.color_palette('viridis', len(top_manufacturers)),
                   edgecolor='white')
    
    for bar, val in zip(bars, top_manufacturers['avg_speed']):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f} mph', va='center', fontsize=10)
    
    ax.set_title('Топ-15 виробників за середньою швидкістю гірок (мін. 5 гірок)', 
                 fontsize=13, pad=12)
    ax.set_xlabel('Середня швидкість (mph)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

---
## Крок 7: Виявлення та обробка викидів (Outliers)

**Навіщо?** Викиди — це значення, що суттєво відрізняються від решти даних. Вони можуть:
- бути помилками вимірювання (погані дані)
- бути реальними екстремальними значеннями (рекордна гірка)
- суттєво спотворювати середнє та результати ML-моделей

**Метод IQR (Interquartile Range):**  
Викид = значення нижче `Q1 - 1.5×IQR` або вище `Q3 + 1.5×IQR`

**Z-score метод:**  
Викид = значення, де |z-score| > 3

In [ ]:
def detect_outliers_iqr(series):
    """Виявляє викиди методом IQR. Повертає маску True/False."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return (series < lower) | (series > upper), lower, upper

# Аналіз викидів для ключових колонок
outlier_cols = ['speed_mph', 'height_ft', 'Gforce_clean']
outlier_cols = [c for c in outlier_cols if c in df_clean.columns]

print('Аналіз викидів (метод IQR):')
print('-' * 60)
for col in outlier_cols:
    mask, lower, upper = detect_outliers_iqr(df_clean[col].dropna())
    n_out = mask.sum()
    pct = n_out / len(mask) * 100
    print(f'{col:<20}: {n_out:>4} викидів ({pct:.1f}%) | діапазон: [{lower:.1f}, {upper:.1f}]')

In [ ]:
# Boxplot для виявлення викидів
fig, axes = plt.subplots(1, len(outlier_cols), figsize=(16, 6))
if len(outlier_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, outlier_cols):
    data = df_clean[col].dropna()
    mask, lower, upper = detect_outliers_iqr(data)
    
    bp = ax.boxplot(data, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightsteelblue', color='navy'),
                    flierprops=dict(marker='o', color='red', markersize=5, alpha=0.6),
                    medianprops=dict(color='red', linewidth=2))
    
    n_outliers = mask.sum()
    ax.set_title(f'{col}\nВикидів: {n_outliers}', fontsize=12)
    ax.set_ylabel(col)
    ax.set_xticklabels([])

plt.suptitle('Boxplot: виявлення викидів (червоні точки = викиди)', fontsize=13)
plt.tight_layout()
plt.show()

---
## Крок 8: Висновки та інсайти

**Навіщо?** Фінальний крок EDA — систематизувати всі знахідки та сформулювати висновки. Хороший аналіз завжди завершується чіткими відповідями на питання: *Що ми дізналися? Що цікавого знайшли? Що потрібно дослідити далі?*

In [ ]:
# Фінальний дашборд — ключові метрики датасету
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Підсумковий дашборд: Roller Coaster Database EDA', fontsize=16, y=1.01)

# 1. Топ-5 країн за кількістю гірок
ax1 = axes[0, 0]
if 'Location' in df_clean.columns:
    top_locations = df_clean['Location'].value_counts().head(10)
    colors1 = sns.color_palette('Blues_r', len(top_locations))
    top_locations.plot(kind='barh', ax=ax1, color=colors1, edgecolor='white')
    ax1.set_title('Топ-10 локацій за к-стю гірок', fontsize=12)
    ax1.set_xlabel('Кількість')
    ax1.invert_yaxis()

# 2. Розподіл типів гірок (pie chart)
ax2 = axes[0, 1]
if 'Type_Main' in df_clean.columns:
    type_counts = df_clean['Type_Main'].value_counts()
    wedge_props = dict(width=0.5, edgecolor='white', linewidth=2)
    ax2.pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(type_counts)),
            wedgeprops=wedge_props, startangle=90)
    ax2.set_title('Розподіл за типом гірки', fontsize=12)

# 3. Швидкість по десятиріччях
ax3 = axes[1, 0]
if 'year_introduced' in df_clean.columns and 'speed_mph' in df_clean.columns:
    df_clean['decade'] = (df_clean['year_introduced'] // 10 * 10).astype(int)
    decade_speed = df_clean.groupby('decade')['speed_mph'].median()
    decade_speed.plot(kind='bar', ax=ax3, color='coral', edgecolor='white')
    ax3.set_title('Медіанна швидкість гірок по десятиріччях', fontsize=12)
    ax3.set_xlabel('Десятиріччя')
    ax3.set_ylabel('Медіана швидкості (mph)')
    ax3.tick_params(axis='x', rotation=45)

# 4. К-сть інверсій vs G-force
ax4 = axes[1, 1]
if 'Inversions_clean' in df_clean.columns and 'Gforce_clean' in df_clean.columns:
    inv_gforce = df_clean.groupby('Inversions_clean')['Gforce_clean'].agg(['mean', 'count']).reset_index()
    inv_gforce = inv_gforce[inv_gforce['count'] >= 5]
    scatter = ax4.scatter(inv_gforce['Inversions_clean'], inv_gforce['mean'],
                          s=inv_gforce['count'] * 5, alpha=0.7, 
                          c=inv_gforce['mean'], cmap='YlOrRd')
    ax4.set_title('Інверсії vs середній G-force\n(розмір = к-сть гірок)', fontsize=12)
    ax4.set_xlabel('Кількість інверсій')
    ax4.set_ylabel('Середній G-force')
    plt.colorbar(scatter, ax=ax4, label='G-force')

plt.tight_layout()
plt.show()

---
## 🛠️ Бонус: Корисні однорядкові функції для студентів

Ці функції — маленькі помічники, які економлять час при щоденній роботі з даними. Зберіть їх собі у власну бібліотеку!

In [ ]:
# ============================================================
# КОРИСНІ ОДНОРЯДКОВІ ФУНКЦІЇ ДЛЯ EDA
# ============================================================

# --- ОГЛЯД ДАНИХ ---

# Швидкий огляд датасету: розмір + типи + пропуски + приклад
quick_look = lambda df: print(f"Розмір: {df.shape} | Типи: {dict(df.dtypes.value_counts())} | Пропуски: {df.isnull().sum().sum()}")

# Відсоток пропущених значень по кожній колонці (тільки ті, де є пропуски)
missing_pct = lambda df: (df.isnull().mean() * 100).round(2).sort_values(ascending=False)[lambda x: x > 0]

# Кількість дублікатів у датафреймі
count_dupes = lambda df: df.duplicated().sum()

# Унікальні значення для всіх категоріальних колонок
cat_uniques = lambda df: {col: df[col].unique().tolist() for col in df.select_dtypes('object').columns}

# Частка кожної категорії у колонці (у %)
value_pct = lambda df, col: (df[col].value_counts(normalize=True) * 100).round(2)

# --- ЧИСЛОВІ ДАНІ ---

# Коефіцієнт варіації (відносне розсіювання) — що менше, то стабільніше
cv = lambda series: round(series.std() / series.mean() * 100, 2)

# Перцентилі колонки: 1%, 5%, 25%, 50%, 75%, 95%, 99%
percentiles = lambda series: series.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

# Нижня та верхня межа викидів (IQR метод)
iqr_bounds = lambda s: (s.quantile(0.25) - 1.5 * (s.quantile(0.75) - s.quantile(0.25)),
                        s.quantile(0.75) + 1.5 * (s.quantile(0.75) - s.quantile(0.25)))

# Кількість викидів у колонці (IQR)
count_outliers = lambda s: ((s < s.quantile(0.25) - 1.5*(s.quantile(0.75)-s.quantile(0.25))) |
                            (s > s.quantile(0.75) + 1.5*(s.quantile(0.75)-s.quantile(0.25)))).sum()

# Z-score для серії даних
zscore = lambda series: (series - series.mean()) / series.std()

# Нормалізація в діапазон [0, 1] (min-max scaling)
normalize = lambda series: (series - series.min()) / (series.max() - series.min())

# Стандартизація (z-score scaling)
standardize = lambda series: (series - series.mean()) / series.std()

# --- ФІЛЬТРАЦІЯ ---

# Рядки, де значення у колонці > N стандартних відхилень від середнього
extreme_rows = lambda df, col, n=3: df[df[col].apply(lambda x: abs((x - df[col].mean()) / df[col].std()) > n)]

# Вибір лише числових колонок
num_cols = lambda df: df.select_dtypes(include='number').columns.tolist()

# Вибір лише категоріальних колонок
cat_cols_fn = lambda df: df.select_dtypes(include='object').columns.tolist()

# Колонки з пропусками (список)
cols_with_na = lambda df: df.columns[df.isnull().any()].tolist()

# --- КОРЕЛЯЦІЇ ---

# Топ-N найбільших кореляцій (за абсолютним значенням)
top_corr = lambda df, n=10: (df.select_dtypes('number').corr().abs()
                              .unstack().sort_values(ascending=False)
                              .drop_duplicates()[lambda x: x < 1].head(n))